# Complete Setup Diagnostics

Use this notebook to verify the project environment before running the Data Governance Copilot.

It checks:

- Python executable, version, and working directory
- Project root and `src` import path
- `.env` loading without printing secrets
- Required and optional library imports with versions
- Core project module imports
- App config sanity
- Redis connectivity
- SQLite memory path
- Docker Compose service status, when Docker is available
- A final pass/fail summary

In [1]:
from pathlib import Path
import importlib
import importlib.metadata as md
import os
import platform
import socket
import subprocess
import sys
import traceback

checks = []

def record(name, ok, detail=""):
    status = "PASS" if ok else "FAIL"
    checks.append({"check": name, "ok": bool(ok), "detail": str(detail)})
    print(f"[{status}] {name}: {detail}")

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())

record("Python >= 3.12", sys.version_info >= (3, 12), platform.python_version())

Python executable: d:\0_PROJECTS\data-governance-copilot\.venv\Scripts\python.exe
Python version: 3.12.12 (main, Jan 13 2026, 17:36:25) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
Working directory: d:\0_PROJECTS\data-governance-copilot\notebook\phase
[PASS] Python >= 3.12: 3.12.12


## 1. Locate Project Root

In [ ]:
cwd = Path.cwd().resolve()
project_root = None

for candidate in [cwd, *cwd.parents]:
    if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
        project_root = candidate
        break

if project_root is None:
    project_root = cwd

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project_root:", project_root)
print("src_path:", src_path)
print("src exists:", src_path.exists())
print("pyproject exists:", (project_root / "pyproject.toml").exists())

record("Project root found", src_path.exists() and (project_root / "pyproject.toml").exists(), project_root)

## 2. Environment File Check

This cell reports whether `.env` exists and lists key names only. It does not print secret values.

In [ ]:
env_path = project_root / ".env"
example_path = project_root / ".env.example"

print(".env exists:", env_path.exists())
print(".env.example exists:", example_path.exists())

interesting_prefixes = (
    "ENABLE_MOCK", "DEBUG", "LOG_LEVEL", "LLM_PROVIDER", "LLM_MODEL",
    "GROQ_API_KEY", "OPENAI_API_KEY", "REDIS_", "SQLITE_PATH",
    "DATABRICKS_", "JIRA_", "LANGCHAIN_", "LANGSMITH_",
)

if env_path.exists():
    visible_keys = []
    for raw_line in env_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key = line.split("=", 1)[0].strip()
        if key.startswith(interesting_prefixes):
            visible_keys.append(key)
    print("configured keys:")
    for key in sorted(set(visible_keys)):
        print(" -", key)

record(".env or .env.example present", env_path.exists() or example_path.exists(), ".env controls runtime config")

## 3. Package Versions

This checks the main runtime dependencies. Missing optional packages are marked as warnings in the detail, not hard failures.

In [ ]:
packages = {
    "python-dotenv": "dotenv",
    "requests": "requests",
    "pydantic": "pydantic",
    "langchain": "langchain",
    "langchain-core": "langchain_core",
    "langchain-community": "langchain_community",
    "langchain-openai": "langchain_openai",
    "langchain-groq": "langchain_groq",
    "langchain-litellm": "langchain_litellm",
    "langgraph": "langgraph",
    "langgraph-checkpoint-sqlite": "langgraph.checkpoint.sqlite",
    "redis": "redis",
    "hiredis": "hiredis",
    "streamlit": "streamlit",
    "fastapi": "fastapi",
    "uvicorn": "uvicorn",
    "slowapi": "slowapi",
    "sse-starlette": "sse_starlette",
    "pytest": "pytest",
}

missing = []
for dist_name, import_name in packages.items():
    try:
        module = importlib.import_module(import_name)
        try:
            version = md.version(dist_name)
        except md.PackageNotFoundError:
            version = getattr(module, "__version__", "imported")
        print(f"{dist_name:32s} {version}")
    except Exception as exc:
        missing.append((dist_name, str(exc)))
        print(f"{dist_name:32s} MISSING ({exc})")

required = [
    "python-dotenv", "requests", "pydantic", "langchain", "langchain-core",
    "langgraph", "redis", "streamlit",
]
missing_required = [name for name, _ in missing if name in required]
record("Required packages import", not missing_required, missing_required or "all required imports ok")

## 4. LangChain Version Matrix

This checks the versions that commonly break when mixed across major lines.

In [ ]:
def version_or_none(dist_name):
    try:
        return md.version(dist_name)
    except md.PackageNotFoundError:
        return None

version_names = [
    "langchain", "langchain-core", "langchain-community", "langchain-openai",
    "langchain-groq", "langchain-litellm", "langgraph",
]
versions = {name: version_or_none(name) for name in version_names}
for name, version in versions.items():
    print(f"{name:28s} {version}")

core_ok = bool(versions.get("langchain-core", "").startswith("0.3."))
lc_ok = bool(versions.get("langchain", "").startswith("0.3."))
groq_ok = versions.get("langchain-groq") is None or versions["langchain-groq"].startswith("0.3.")
record("LangChain 0.3.x compatibility", core_ok and lc_ok and groq_ok, versions)

## 5. Project Module Imports

In [ ]:
project_modules = [
    "config.settings",
    "core.cache",
    "core.llm_factory",
    "core.base_agent",
    "core.guardrails",
    "graph.state",
    "graph.intent",
    "graph.routing",
    "graph.nodes",
    "memory.checkpointer",
    "agents.information_agent",
    "agents.knowledge_agent",
    "agents.metadata_agent",
    "agents.capacity_agent",
    "agents.rule_agent",
]

failed_imports = []
for module_name in project_modules:
    try:
        importlib.import_module(module_name)
        print("OK", module_name)
    except Exception as exc:
        failed_imports.append((module_name, repr(exc)))
        print("FAIL", module_name, repr(exc))

record("Project modules import", not failed_imports, failed_imports or "all imports ok")

## 6. App Config Sanity

Secrets are not printed. This confirms config objects load and key values are reasonable.

In [ ]:
from config.settings import AppConfig, DATA_PRODUCTS

app_config = AppConfig()

print("debug:", app_config.debug)
print("log_level:", app_config.log_level)
print("enable_mock:", app_config.enable_mock)
print("llm_provider:", app_config.llm.provider)
print("llm_model:", app_config.llm.primary_model)
print("llm_api_key_set:", bool(app_config.llm.api_key))
print("redis_enabled:", app_config.redis.enabled)
print("redis_host:", app_config.redis.host)
print("redis_port:", app_config.redis.port)
print("data_products:", sorted(DATA_PRODUCTS.keys()))

record("Data products registered", bool(DATA_PRODUCTS), sorted(DATA_PRODUCTS.keys()))
record("LLM configured or mock mode", app_config.enable_mock or bool(app_config.llm.api_key), "mock mode can run without real keys")

## 7. Redis Connectivity

In [ ]:
import core.cache as cache
from core.cache import get_client, cache_set, cache_get

cache._client = None
redis_client = get_client(app_config.redis)

if redis_client:
    test_key = "setup_diagnostics:ping"
    cache_set(redis_client, test_key, {"ok": True}, ttl=60)
    readback = cache_get(redis_client, test_key)
    redis_client.delete(test_key)
    print("redis_url:", app_config.redis.url.replace(app_config.redis.password, "<redacted>") if app_config.redis.password else app_config.redis.url)
    print("ping:", redis_client.ping())
    print("readback:", readback)
    record("Redis live connection", readback == {"ok": True}, "round trip ok")
else:
    detail = "Redis unavailable; app will use in-memory fallback"
    print(detail)
    record("Redis live connection", False, detail)

## 8. SQLite Memory Path

In [ ]:
sqlite_path = os.getenv("SQLITE_PATH", "./data/memory.db")
sqlite_file = (project_root / sqlite_path).resolve() if not Path(sqlite_path).is_absolute() else Path(sqlite_path)

print("SQLITE_PATH:", sqlite_path)
print("resolved:", sqlite_file)
print("parent exists:", sqlite_file.parent.exists())
print("db exists:", sqlite_file.exists())

record("SQLite parent directory exists", sqlite_file.parent.exists(), sqlite_file.parent)

## 9. Docker Compose Status

This is optional. It works only when Docker is available from the notebook environment.

In [ ]:
def run_cmd(cmd, timeout=20):
    try:
        result = subprocess.run(
            cmd,
            cwd=project_root,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        return result.returncode, result.stdout.strip(), result.stderr.strip()
    except Exception as exc:
        return 999, "", repr(exc)

code, out, err = run_cmd(["docker", "compose", "ps"])
if code == 0:
    print(out)
    record("Docker Compose reachable", True, "docker compose ps succeeded")
else:
    print("Docker Compose unavailable or blocked")
    print(err)
    record("Docker Compose reachable", False, err or "non-zero exit")

## 10. Optional Graph Import/Compile Check

This imports the compiled graph. It should not run a user query, but it may initialize the checkpointer and agent singletons.

In [ ]:
RUN_GRAPH_COMPILE_CHECK = True

if RUN_GRAPH_COMPILE_CHECK:
    try:
        from graph.graph import copilot_graph
        print("copilot_graph:", type(copilot_graph))
        record("LangGraph compiles/imports", True, type(copilot_graph))
    except Exception as exc:
        traceback.print_exc()
        record("LangGraph compiles/imports", False, repr(exc))
else:
    print("Skipped. Set RUN_GRAPH_COMPILE_CHECK = True to run this cell.")

## 11. Final Summary

In [ ]:
passed = sum(1 for item in checks if item["ok"])
failed = [item for item in checks if not item["ok"]]

print(f"Passed: {passed}/{len(checks)}")

if failed:
    print("\nFailures / warnings to fix:")
    for item in failed:
        print(f"- {item['check']}: {item['detail']}")
else:
    print("All diagnostics passed.")

# Keep this assert commented if you want the notebook to complete even with optional warnings.
# assert not failed